# 07_event_based_device_control: Notebook Scaffold and Device-Level Data Pipeline

This notebook establishes the complete data pipeline from raw CSVs through 15-minute aligned device-level and room-level features for event-based control.

In [ ]:
import os
import sys
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
%matplotlib inline

# Resolve backend/notebooks directory for imports
sys.path.append(os.path.abspath('.'))
from utils import (
    load_processed,
    load_space_events,
    build_device_level_features,
    build_event_features,
    build_room_level_features,
    save_model_artifacts,
    load_model_artifacts,
)

# Flag to load existing model artifacts
LOAD_EXISTING_MODEL = True


## 1. Configurations

In [ ]:
comfort_config = {
    't_min_occupied_celsius': 20.0,
    't_max_occupied_celsius': 24.0,
    't_min_unoccupied_celsius': 16.0,
    't_max_unoccupied_celsius': 28.0,
    'relative_humidity_min_percent': 30.0,
    'relative_humidity_max_percent': 60.0,
}

control_config = {
    'preconditioning_window_hours': 2.0,
    'standby_baseline_heating': 1.2,
    'standby_baseline_cooling': 2.0,
}

energy_config = {
    'electricity_price_eur_per_kWh': 0.35,
    'co2_kg_per_kWh': 0.40,
}

## 2. Data Loading & Feature Engineering

In [ ]:
# Load seasonal processed datasets
df_heating_raw = load_processed('heating')
df_cooling_raw = load_processed('cooling')

# Load space events
events_heating = load_space_events('heating')
events_cooling = load_space_events('cooling')

# Build device-level features (wide format)
df_dev_heating = build_device_level_features(
    df_heating_raw, standby_baseline=control_config['standby_baseline_heating']
)
df_dev_cooling = build_device_level_features(
    df_cooling_raw, standby_baseline=control_config['standby_baseline_cooling']
)

# Build room-level features
df_room_heating = build_room_level_features(df_heating_raw)
df_room_cooling = build_room_level_features(df_cooling_raw)

# Build event features
df_evt_heating = build_event_features(
    df_room_heating.index,
    events_heating,
    preconditioning_window_hours=control_config['preconditioning_window_hours'],
)
df_evt_cooling = build_event_features(
    df_room_cooling.index,
    events_cooling,
    preconditioning_window_hours=control_config['preconditioning_window_hours'],
)

# Merge all features
df_heating_final = pd.concat([df_room_heating, df_dev_heating, df_evt_heating], axis=1)
df_cooling_final = pd.concat([df_room_cooling, df_dev_cooling, df_evt_cooling], axis=1)

print('Heating Final Data Shape:', df_heating_final.shape)
print('Cooling Final Data Shape:', df_cooling_final.shape)

## 3. Data Quality Summary

In [ ]:
def print_quality_summary(df_raw, df_final, events, name):
    print(f'=== {name.upper()} SEASON ===')
    print(f'Raw row count: {len(df_raw)}')
    print(f'Final aligned row count (15-min): {len(df_final)}')
    print(f'Unique devices: {df_raw["device_id"].nunique()}')
    print(f'Event counts: {len(events)}')
    print('Missing value statistics in critical fields:')
    critical_cols = [
        'room_temp', 'room_return_temp', 'room_humidity', 'room_co2',
        'room_power_kw', 'outside_temperature', 'event_active'
    ]
    for col in critical_cols:
        missing = df_final[col].isna().sum()
        print(f'  {col}: {missing} missing ({missing / len(df_final) * 100:.2f}%)')
    print()

print_quality_summary(df_heating_raw, df_heating_final, events_heating, 'heating')
print_quality_summary(df_cooling_raw, df_cooling_final, events_cooling, 'cooling')

## 4. Grey-Box Thermal Model Training & Validation

We fit a grey-box thermal model of the room temperature dynamics for both heating and cooling seasons.

In [ ]:
import os
import joblib
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from utils import train_thermal_model, predict_temperature, train_device_power_model, predict_device_power

# Splits
split_heat = pd.Timestamp("2026-04-03 00:00:00")
split_cool = pd.Timestamp("2026-05-28 00:00:00")

print(f"Heating split timestamp: {split_heat}")
print(f"Cooling split timestamp: {split_cool}")

artifacts_dir = "../artifacts/event_device_control/models"
required_files = [
    "thermal_model.joblib",
    "scaler.joblib",
    "feature_columns.json",
    "model_metadata.json",
    "device_power_model.joblib",
    "heating_thermal_model.joblib",
    "heating_scaler.joblib",
    "heating_feature_columns.json",
    "heating_model_metadata.json",
    "heating_device_power_model.joblib",
    "cooling_thermal_model.joblib",
    "cooling_scaler.joblib",
    "cooling_feature_columns.json",
    "cooling_model_metadata.json",
    "cooling_device_power_model.joblib",
]
all_exist = all(os.path.exists(os.path.join(artifacts_dir, f)) for f in required_files)

df_h = df_heating_final.copy()
df_c = df_cooling_final.copy()

if 'LOAD_EXISTING_MODEL' in globals() and LOAD_EXISTING_MODEL and all_exist:
    print("Loading existing models from disk...")
    model_h, model_power_h = load_model_artifacts(artifacts_dir, "heating")
    model_c, model_power_c = load_model_artifacts(artifacts_dir, "cooling")
else:
    print("Training models from scratch...")
    # Train heating thermal model
    print("\n--- Training Heating Thermal Model ---")
    model_h = train_thermal_model(df_h, "heating", split_heat)
    
    # Train cooling thermal model
    print("\n--- Training Cooling Thermal Model ---")
    model_c = train_thermal_model(df_c, "cooling", split_cool)
    
    # Train heating device power model
    print("\n--- Training Heating Device Power Model ---")
    model_power_h = train_device_power_model(df_h, "heating", split_heat)
    
    # Train cooling device power model
    print("\n--- Training Cooling Device Power Model ---")
    model_power_c = train_device_power_model(df_c, "cooling", split_cool)

print("\n--- Heating Thermal Model Coefficients ---")
print(f"Is fallback: {model_h['is_fallback']}")
print(f"a = {model_h['a']:.6f}")
print(f"b = {model_h['b']:.6f}")
print(f"c_heat = {model_h['c_heat']:.6f}")
print(f"d = {model_h['d']:.6f}")
print(f"intercept = {model_h['intercept']:.6f}")

print("\n--- Cooling Thermal Model Coefficients ---")
print(f"Is fallback: {model_c['is_fallback']}")
print(f"a = {model_c['a']:.6f}")
print(f"b = {model_c['b']:.6f}")
print(f"c_cool = {model_c['c_cool']:.6f}")
print(f"d = {model_c['d']:.6f}")
print(f"intercept = {model_c['intercept']:.6f}")

print("\n--- Heating Device Power Model Parameters ---")
for dev, params in model_power_h["device_params"].items():
    print(f"  {dev}: active_power = {params['active_power']:.4f} kW, standby_power = {params['standby_power']:.4f} kW")

print("\n--- Cooling Device Power Model Parameters ---")
for dev, params in model_power_c["device_params"].items():
    print(f"  {dev}: active_power = {params['active_power']:.4f} kW, standby_power = {params['standby_power']:.4f} kW")


### Validation & Accuracy Metrics

In [ ]:
# Validation split metrics
val_h = df_h[df_h.index >= split_heat].copy()
val_h['T_next'] = val_h['room_temp'].shift(-1)
val_h = val_h.dropna(subset=['T_next'])
preds_h = predict_temperature(model_h, val_h)
mae_h = mean_absolute_error(val_h['T_next'], preds_h)
rmse_h = root_mean_squared_error(val_h['T_next'], preds_h)

val_c = df_c[df_c.index >= split_cool].copy()
val_c['T_next'] = val_c['room_temp'].shift(-1)
val_c = val_c.dropna(subset=['T_next'])
preds_c = predict_temperature(model_c, val_c)
mae_c = mean_absolute_error(val_c['T_next'], preds_c)
rmse_c = root_mean_squared_error(val_c['T_next'], preds_c)

print("HEATING VALIDATION METRICS:")
print(f"  MAE: {mae_h:.6f} °C")
print(f"  RMSE: {rmse_h:.6f} °C")

print("\nCOOLING VALIDATION METRICS:")
print(f"  MAE: {mae_c:.6f} °C")
print(f"  RMSE: {rmse_c:.6f} °C")

# Check validation MAE boundary
assert mae_h < 2.0, f"Heating MAE too high: {mae_h:.4f}"
assert mae_c < 2.0, f"Cooling MAE too high: {mae_c:.4f}"
print("\nValidation bounds check passed (MAE < 2.0 °C)!")

# Evaluate device power model predictions on validation set
import numpy as np
print("\nHEATING DEVICE POWER RMSE:")
preds_power_h = predict_device_power(model_power_h, val_h)
for i in range(1, 5):
    actual = val_h[f"device_{i}_power_kw"]
    pred = preds_power_h[f"device_{i}_predicted_power_kw"]
    mask = actual.notna() & pred.notna()
    rmse = np.sqrt(np.mean((actual[mask] - pred[mask])**2)) if mask.any() else 0.0
    print(f"  device_{i} RMSE: {rmse:.6f} kW")

print("\nCOOLING DEVICE POWER RMSE:")
preds_power_c = predict_device_power(model_power_c, val_c)
for i in range(1, 5):
    actual = val_c[f"device_{i}_power_kw"]
    pred = preds_power_c[f"device_{i}_predicted_power_kw"]
    mask = actual.notna() & pred.notna()
    rmse = np.sqrt(np.mean((actual[mask] - pred[mask])**2)) if mask.any() else 0.0
    print(f"  device_{i} RMSE: {rmse:.6f} kW")

### Save & Reload Verification

In [ ]:
# Save model artifacts using helper function if they were not loaded
artifacts_dir = "../artifacts/event_device_control/models"
os.makedirs(artifacts_dir, exist_ok=True)

if not ('LOAD_EXISTING_MODEL' in globals() and LOAD_EXISTING_MODEL and all_exist):
    print("Saving model artifacts...")
    save_model_artifacts(
        artifacts_dir=artifacts_dir,
        thermal_model_dict=model_h,
        device_power_model_dict=model_power_h,
        metrics={"mae": mae_h, "rmse": rmse_h},
        train_timestamps={"split": str(split_heat)}
    )
    save_model_artifacts(
        artifacts_dir=artifacts_dir,
        thermal_model_dict=model_c,
        device_power_model_dict=model_power_c,
        metrics={"mae": mae_c, "rmse": rmse_c},
        train_timestamps={"split": str(split_cool)}
    )
    print(f"Saved models to {artifacts_dir}")
else:
    print("Skipping save since existing models were loaded.")

# Load verifying using load_model_artifacts
model_h_reloaded, model_power_h_reloaded = load_model_artifacts(artifacts_dir, "heating")
model_c_reloaded, model_power_c_reloaded = load_model_artifacts(artifacts_dir, "cooling")

# Verify predictions match exactly
preds_h_reloaded = predict_temperature(model_h_reloaded, val_h)
preds_c_reloaded = predict_temperature(model_c_reloaded, val_c)

assert preds_h_reloaded.equals(preds_h), "Heating reloaded model predictions mismatch!"
assert preds_c_reloaded.equals(preds_c), "Cooling reloaded model predictions mismatch!"

preds_power_h_reloaded = predict_device_power(model_power_h_reloaded, val_h)
preds_power_c_reloaded = predict_device_power(model_power_c_reloaded, val_c)

assert preds_power_h_reloaded.equals(preds_power_h), "Heating power reloaded model predictions mismatch!"
assert preds_power_c_reloaded.equals(preds_power_c), "Cooling power reloaded model predictions mismatch!"

print("Model save and reload verification passed successfully!")
